# AWARE VAST read-only preflight and inventory

Run this notebook only after manually selecting the existing VAST kernel in VS Code. Review every cell before running it. For the July 22 preflight, run only these cells: they validate paths and inspect metadata without creating, downloading, moving, deleting, training, or exporting anything.

Required environment variables: `PROJECT_CODE_ROOT`, `PROJECT_DATA_ROOT`, and `PROJECT_OUTPUT_ROOT`. Their values stay in the remote environment and must not be pasted into committed files or returned in an unredacted report.

In [ ]:
from pathlib import Path
import os
import socket
import sys

required = ("PROJECT_CODE_ROOT", "PROJECT_DATA_ROOT", "PROJECT_OUTPUT_ROOT")
print("hostname:", socket.gethostname())
print("python:", sys.executable)
print("cwd:", Path.cwd())
for name in required:
    value = os.environ.get(name, "").strip()
    print(name, "set=", bool(value), "absolute=", bool(value and Path(value).is_absolute()))
    if not value:
        raise RuntimeError(f"{name} is not set; stop and configure the remote kernel environment")

In [ ]:
code_root = Path(os.environ["PROJECT_CODE_ROOT"]).resolve()
if Path.cwd().resolve() != code_root:
    raise RuntimeError("Kernel cwd does not equal PROJECT_CODE_ROOT; stop without changing directories")
for relative in ("src", "configs", "ontology.yaml", "source_manifest.yaml"):
    if not (code_root / relative).exists():
        raise RuntimeError(f"Required project item is missing: {relative}")

from src.project_paths import ProjectPaths
from src.validation import validate_environment

paths = ProjectPaths.from_environment()
report = validate_environment(paths)
print(report.render())
if not report.ok:
    raise RuntimeError("Environment preflight failed; do not continue")

In [ ]:
from src.metadata_validation import load_yaml_mapping, validate_ontology, validate_source_manifest
from src.mappings import validate_mapping_ledger

ontology = load_yaml_mapping(code_root / "ontology.yaml")
manifest = load_yaml_mapping(code_root / "source_manifest.yaml")
ledger = load_yaml_mapping(code_root / "mapping_ledger.yaml")
results = (
    validate_ontology(ontology),
    validate_source_manifest(manifest, ontology_version=ontology["ontology_version"]),
    validate_mapping_ledger(ledger),
)
for label, result in zip(("ontology", "source manifest", "mapping ledger"), results):
    print(result.render(label))
if not all(result.ok for result in results):
    raise RuntimeError("Metadata validation failed; do not continue")

In [ ]:
from src.inventory import disk_summary, inventory_approved_sources

print(disk_summary(paths.data_root))
inventories = inventory_approved_sources(paths.data_root, manifest)
for inventory in inventories:
    print(inventory.render())
    print()
print("READ-ONLY INVENTORY COMPLETE. Stop here and return a redacted summary for review.")

## Stop gate

Do not download sources, convert labels, create splits, or start training in this notebook until the inventory has been reviewed. Return only: pass/fail status, per-source file/image/annotation counts, extensions, class names, total bytes, free disk space, and errors. Redact private paths, usernames, hostnames, URLs, and credentials.